<a href="https://colab.research.google.com/github/gautamthampy/CMPE256-Group10/blob/baseline-covisitation/RecSysProjject.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# 1. Setup & imports

import math
import random
from collections import Counter, defaultdict
import numpy as np

In [2]:
# 2. Load data

data = {}

file_path = "/content/train-2.txt"

with open(file_path, "r") as f:
    for line in f:
        parts = line.strip().split()
        if not parts:
            continue
        user = parts[0]
        items = parts[1:]
        data[user] = items

print("Number of users:", len(data))

Number of users: 52643


In [3]:
from collections import Counter

item_counts = Counter()
for items in data.values():
    item_counts.update(items)

print("Total unique items:", len(item_counts))
print("Example user, items:", next(iter(data.items())))

Total unique items: 91599
Example user, items: ('0', ['28261', '388', '5731', '401', '28284', '3570', '26806', '6802', '12212', '407', '22036', '29781', '16', '1789', '385', '29376', '661'])


In [4]:
# 3. Build train/validation split

random.seed(42)

train_data = {}  # user -> list of items (without held out)
val_items   = {}  # user -> single held-out item (str)

for user, items in data.items():
    if len(items) == 0:
        continue
    held_out = random.choice(items)
    remaining = [it for it in items if it != held_out]
    # if user had duplicates of held_out, they all get removed; fine for implicit data

    train_data[user] = remaining
    val_items[user] = held_out

print("Users in train_data:", len(train_data))
print("Users in val_items:", len(val_items))

Users in train_data: 52643
Users in val_items: 52643


In [6]:
# 4. NDCG@K for single held-out item per user

def ndcg_at_k_single(recommended, true_item, k=20):
    """
    recommended: list of item ids, ranked from best to worst
    true_item: the held-out item id (string)
    """
    try:
        rank = recommended.index(true_item)
    except ValueError:
        return 0.0

    if rank >= k:
        return 0.0

    # DCG with relevance 1 at position rank
    return 1.0 / math.log2(rank + 2)  # +2 because ranks are 0-based


def mean_ndcg_at_k(recommender_fn, users, val_dict, k=20):
    """
    recommender_fn(user, k) -> list of item_ids
    users: iterable of user ids
    val_dict: dict user -> true_item
    """
    scores = []
    for u in users:
        true_item = val_dict[u]
        recs = recommender_fn(u, k=k)
        scores.append(ndcg_at_k_single(recs, true_item, k))
    return sum(scores) / len(scores)

Algorithm 1: The popularity-based recommender serves as our simplest baseline model. It ranks all items globally by how frequently they appear in the training data and recommends the top-20 most popular items that a user has not already interacted with. Because it does not use any personalization or user-specific signals, every user receives nearly the same recommendations aside from items they have already seen. While extremely fast and easy to implement, this approach performs poorly on ranking metrics, as it generally fails to retrieve each user’s unique held-out item. In our leave-one-out evaluation, the popularity model achieved an NDCG@20 of 0.0047, confirming that


In [7]:
# 5. Popularity baseline

pop_counts = Counter()
for items in train_data.values():
    pop_counts.update(items)

print("Unique items in train:", len(pop_counts))
print("Top 5 most popular items:", pop_counts.most_common(5))

# Global ranked list of items by popularity
items_by_pop = [it for it, _ in pop_counts.most_common()]

Unique items in train: 91569
Top 5 most popular items: [('896', 1681), ('43756', 1352), ('7897', 1207), ('36454', 1123), ('36477', 1098)]


In [8]:
def recommend_popularity(user, k=20):
    """
    Recommend top-k most popular items the user has NOT seen in train_data.
    """
    seen = set(train_data[user])  # items user has interacted with in train set
    recs = []
    for it in items_by_pop:
        if it in seen:
            continue
        recs.append(it)
        if len(recs) == k:
            break
    return recs

In [9]:
users_list = list(train_data.keys())

pop_ndcg_20 = mean_ndcg_at_k(recommend_popularity, users_list, val_items, k=20)
print("Popularity NDCG@20:", pop_ndcg_20)

Popularity NDCG@20: 0.0047054802836267616


Algorithm 2: The co-visitation model is an item-based collaborative filtering approach that recommends items frequently co-occurring with those a user has already interacted with. We build a co-occurrence matrix by counting how often pairs of items appear together across users, applying trimming and neighbor limits to keep the model efficient. For a given user, candidate items are scored based on how strongly they co-occur with the user’s seen items, with a normalization step to reduce popularity bias. This model captures item–item relationships and provides personalized recommendations based on users’ historical interactions. In evaluation, the co-visitation approach significantly outperformed the popularity baseline, achieving an NDCG@20 of approximately 0.083, indicating meaningful improvements in ranking the user’s held-out item.

In [10]:
# 6. Co-visitation model (item-based)

def build_covisit_model(train_data, max_items_per_user=80, max_neighbors=400):
    """
    Returns:
      cooc: dict item_i -> dict item_j -> cooc_count
      pop_counts: Counter of item frequencies in train_data
    """
    # global popularity for trimming
    pop_counts = Counter()
    for items in train_data.values():
        pop_counts.update(items)

    # sort items by global popularity for each user and trim
    pop_rank = {item: rank for rank, (item, _) in enumerate(pop_counts.most_common())}

    trimmed_users = {}
    for user, items in train_data.items():
        # sort this user's items by popularity (most popular first)
        sorted_items = sorted(items, key=lambda it: pop_rank[it])
        trimmed_users[user] = sorted_items[:max_items_per_user]

    cooc = defaultdict(lambda: defaultdict(int))

    # build co-occurrence counts
    for items in trimmed_users.values():
        uniq = list(set(items))
        n = len(uniq)
        for i_idx in range(n):
            for j_idx in range(i_idx + 1, n):
                i = uniq[i_idx]
                j = uniq[j_idx]
                cooc[i][j] += 1
                cooc[j][i] += 1

    # prune neighbors
    for i, neighbors in list(cooc.items()):
        if len(neighbors) > max_neighbors:
            # keep only the most co-visited neighbors
            top_neighbors = sorted(neighbors.items(), key=lambda x: x[1], reverse=True)[:max_neighbors]
            cooc[i] = dict(top_neighbors)

    return cooc, pop_counts

In [11]:
cooc, pop_counts_covisit = build_covisit_model(train_data,
                                              max_items_per_user=80,
                                              max_neighbors=400)
print("Co-visitation model built. Example item neighbors:")
example_item = next(iter(cooc.keys()))
print("Item:", example_item, "neighbors count:", len(cooc[example_item]))

Co-visitation model built. Example item neighbors:
Item: 12212 neighbors count: 400


In [13]:
def recommend_covisit(user, k=20):
    """
    Recommend items using co-visitation model for this user.
    """
    seen = set(train_data[user])
    scores = defaultdict(float)

    for i in seen:
        if i not in cooc:
            continue
        for j, c in cooc[i].items():
            if j in seen:
                continue
            # normalized co-occurrence
            scores[j] += c / math.sqrt(pop_counts_covisit[i] * pop_counts_covisit[j])

    if not scores:
        # fallback to popularity baseline
        return recommend_popularity(user, k=k)

    ranked = sorted(scores.items(), key=lambda x: x[1], reverse=True)
    recs = [it for it, _ in ranked[:k]]

    # If fewer than k recs, pad using popularity
    if len(recs) < k:
        extra = recommend_popularity(user, k=k + 50)
        for it in extra:
            if it not in seen and it not in recs:
                recs.append(it)
                if len(recs) == k:
                    break

    return recs

In [15]:
covisit_ndcg_20 = mean_ndcg_at_k(recommend_covisit, users_list, val_items, k=20)
print("Co-visitation NDCG@20:", covisit_ndcg_20)

Co-visitation NDCG@20: 0.08283013740465046


In [16]:
# 7. Build final co-visitation model on full data (no holdout)

cooc_full, pop_counts_full = build_covisit_model(
    data,
    max_items_per_user=80,
    max_neighbors=400
)

# Popularity list on full data
items_by_pop_full = [it for it, _ in pop_counts_full.most_common()]

In [17]:
def recommend_popularity_full(user, k=20):
    seen = set(data[user])
    recs = []
    for it in items_by_pop_full:
        if it in seen:
            continue
        recs.append(it)
        if len(recs) == k:
            break
    return recs

def recommend_covisit_full(user, k=20):
    seen = set(data[user])
    scores = defaultdict(float)

    for i in seen:
        if i not in cooc_full:
            continue
        for j, c in cooc_full[i].items():
            if j in seen:
                continue
            scores[j] += c / math.sqrt(pop_counts_full[i] * pop_counts_full[j])

    if not scores:
        return recommend_popularity_full(user, k=k)

    ranked = sorted(scores.items(), key=lambda x: x[1], reverse=True)
    recs = [it for it, _ in ranked[:k]]

    # pad if needed
    if len(recs) < k:
        extra = recommend_popularity_full(user, k=k + 50)
        for it in extra:
            if it not in seen and it not in recs:
                recs.append(it)
                if len(recs) == k:
                    break

    return recs